# Analisis Data Science - Sistema de Emergencias

Este notebook analiza el archivo `mdi_detenidosaprehendidos_pm_2026_enero_abril.xlsx` para alimentar el dashboard KPI del operador.

Objetivos:
- Limpiar y perfilar el dataset.
- Construir KPIs operativos.
- Analizar zonas, horarios, infracciones, armas, edad y georreferenciacion.
- Generar modelos heurísticos de riesgo para apoyar la toma de decisiones.
- Exportar un resumen JSON que el backend puede usar en el dashboard.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import math
from collections import Counter
import matplotlib.pyplot as plt

DATA_PATH = Path('mdi_detenidosaprehendidos_pm_2026_enero_abril.xlsx')
SHEET = '1. Detenidos y Aprehendidos'

df = pd.read_excel(DATA_PATH, sheet_name=SHEET)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

## 1. Limpieza de datos

Se normalizan coordenadas, edad, fechas, hora y textos. El dataset tiene latitud/longitud con coma decimal, por eso se transforman a valores numéricos.

In [ ]:
def norm_text(x):
    if pd.isna(x):
        return 'SIN_DATO'
    s = str(x).strip()
    return s if s else 'SIN_DATO'

def to_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(',', '.')
    try:
        return float(s)
    except Exception:
        return np.nan

def hour_of(x):
    if pd.isna(x):
        return np.nan
    m = re.search(r'(\d{1,2}):', str(x))
    if not m:
        return np.nan
    h = int(m.group(1))
    return h if 0 <= h <= 23 else np.nan

work = df.copy()
work['latitud_num'] = work['latitud'].map(to_num)
work['longitud_num'] = work['longitud'].map(to_num)
work['edad_num'] = pd.to_numeric(work['edad'], errors='coerce')
work['fecha'] = pd.to_datetime(work['fecha_detencion_aprehension'], errors='coerce')
work['hora'] = work['hora_detencion_aprehension'].map(hour_of)
work['mes'] = work['fecha'].dt.to_period('M').astype(str)
work['dia_semana'] = work['fecha'].dt.day_name()
work[['latitud_num','longitud_num','edad_num','fecha','hora','mes']].head()

## 2. KPIs principales

In [ ]:
geo = work[work['latitud_num'].notna() & work['longitud_num'].notna()].copy()
geo = geo[(geo['latitud_num'].between(-5.5, 1.8)) & (geo['longitud_num'].between(-82, -75))]

sin_arma = ['NINGUNA', 'SIN_DATO', 'SE_DESCONOCE']
con_arma = ~work['arma'].map(norm_text).str.upper().isin(sin_arma)

kpis = {
    'total_registros': len(work),
    'registros_georreferenciados': len(geo),
    'porcentaje_georreferenciado': round(len(geo) / len(work) * 100, 2),
    'edad_promedio': round(work['edad_num'].mean(), 1),
    'edad_mediana': round(work['edad_num'].median(), 1),
    'porcentaje_con_arma': round(con_arma.mean() * 100, 2),
    'provincia_mayor_incidencia': work['nombre_provincia'].map(norm_text).value_counts().idxmax(),
    'infraccion_mayor_incidencia': work['presunta_infraccion'].map(norm_text).value_counts().idxmax(),
    'hora_mayor_incidencia': int(work['hora'].value_counts().idxmax()),
}
kpis

## 3. Distribuciones para dashboard

Estos gráficos permiten ver patrones por tipo, infracción, provincia, cantón, distrito, hora y edad.

In [ ]:
def plot_top(col, n=10, title=None):
    data = work[col].map(norm_text).value_counts().head(n).sort_values()
    ax = data.plot(kind='barh', figsize=(9, 5), color='#4F7BFF')
    ax.set_title(title or f'Top {n} - {col}')
    ax.set_xlabel('Registros')
    plt.tight_layout()
    plt.show()

plot_top('presunta_infraccion', 10, 'Top infracciones')
plot_top('nombre_provincia', 10, 'Top provincias')
plot_top('nombre_canton', 10, 'Top cantones')

In [ ]:
hourly = work['hora'].dropna().astype(int).value_counts().reindex(range(24), fill_value=0)
ax = hourly.plot(kind='line', marker='o', figsize=(10,4), color='#EF4444')
ax.set_title('Incidencia por hora')
ax.set_xlabel('Hora')
ax.set_ylabel('Registros')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
age_bins = [0, 17, 29, 44, 59, 120]
age_labels = ['0-17', '18-29', '30-44', '45-59', '60+']
age_group = pd.cut(work['edad_num'], bins=age_bins, labels=age_labels, include_lowest=True)
age_counts = age_group.value_counts().reindex(age_labels, fill_value=0)
age_counts.plot(kind='bar', figsize=(8,4), color='#22C55E', title='Registros por grupo de edad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Conteo de palabras relevantes

Este conteo sirve para alimentar un panel tipo nube de palabras y entender los términos dominantes en las infracciones.

In [ ]:
stop = {'DE','DEL','LA','LAS','LOS','EL','Y','A','POR','PARA','CON','EN','AL','UN','UNA','SIN','SE','NO','O','U','SU','SUS','QUE','DERECHO','DERECHOS','CONTRA'}
words = Counter()
for txt in work['presunta_infraccion'].map(norm_text):
    for w in re.findall(r'[A-ZÁÉÍÓÚÑÜ]{4,}', txt.upper()):
        if w not in stop:
            words[w] += 1
pd.DataFrame(words.most_common(25), columns=['palabra', 'frecuencia'])

## 5. Mapa de calor

Se agrupan coordenadas redondeando latitud/longitud para reducir ruido y construir puntos de calor para el dashboard.

In [ ]:
geo['lat_grid'] = geo['latitud_num'].round(2)
geo['lng_grid'] = geo['longitud_num'].round(2)
heat = geo.groupby(['lat_grid','lng_grid']).size().reset_index(name='conteo').sort_values('conteo', ascending=False)
heat.head(20)

In [ ]:
plt.figure(figsize=(8, 7))
plt.scatter(geo['longitud_num'], geo['latitud_num'], s=1, alpha=0.25, c='#4F7BFF')
plt.title('Distribucion geografica de registros')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.tight_layout()
plt.show()

## 6. Modelo heurístico de riesgo por distrito

Sin instalar librerías externas, se crea un score de riesgo combinando:
- cantidad de registros,
- casos con arma,
- eventos nocturnos.

Este modelo no reemplaza un modelo supervisado, pero es útil como baseline interpretable para el dashboard.

In [ ]:
risk = work.copy()
risk['tiene_arma'] = ~risk['arma'].map(norm_text).str.upper().isin(sin_arma)
risk['nocturno'] = risk['hora'].fillna(-1).between(20,23) | risk['hora'].fillna(-1).between(0,5)

risk_district = risk.groupby('nombre_distrito').agg(
    total=('tipo', 'size'),
    con_arma=('tiene_arma', 'sum'),
    nocturnos=('nocturno', 'sum')
).reset_index()
risk_district['score_riesgo'] = risk_district['total'] + risk_district['con_arma'] * 2 + risk_district['nocturnos'] * 0.5
risk_district.sort_values('score_riesgo', ascending=False).head(15)

## 7. Modelo de clasificación automática de gravedad

Para conectar con el sistema de emergencias, se propone una clasificación de gravedad basada en palabras clave y contexto. Este modelo se puede usar como baseline antes de entrenar modelos supervisados.

In [ ]:
PALABRAS_CRITICAS = ['muerto', 'fallecido', 'sin vida', 'explosion', 'derrumbe', 'atrapado', 'baleado', 'apunalado', 'no respira', 'paro cardiaco']
PALABRAS_ALTAS = ['arma', 'sangre', 'herido', 'fuego', 'incendio', 'colision', 'atropello', 'hemorragia', 'pistola', 'cuchillo']
PALABRAS_MEDIAS = ['choque', 'lesion', 'caida', 'pelea', 'fuga', 'humo', 'dolor fuerte']

def clasificar_gravedad_texto(texto, tipo='otro'):
    texto = norm_text(texto).lower()
    base = {'incendio': 'critica', 'desastre_natural': 'critica', 'robo': 'alta', 'accidente': 'media', 'otro': 'baja'}.get(tipo, 'baja')
    if any(p in texto for p in PALABRAS_CRITICAS):
        return 'critica'
    if any(p in texto for p in PALABRAS_ALTAS):
        return 'alta'
    if base in ['baja'] and any(p in texto for p in PALABRAS_MEDIAS):
        return 'media'
    return base

pruebas = [
    ('me asaltaron con arma y hay sangre', 'robo'),
    ('choque con lesionados en avenida', 'accidente'),
    ('explosion y persona atrapada', 'incendio'),
    ('consulta por ruido', 'otro'),
]
[(txt, clasificar_gravedad_texto(txt, tipo)) for txt, tipo in pruebas]

## 8. Exportacion para backend/dashboard

El JSON generado resume los datos para que la aplicación no tenga que cargar el Excel completo en cada petición.

In [ ]:
def top_counts(col, n=12):
    return [{'label': str(k), 'valor': int(v)} for k, v in work[col].map(norm_text).value_counts().head(n).items()]

summary = {
    'metadata': {'fuente': str(DATA_PATH), 'sheet': SHEET, 'filas': int(len(work)), 'columnas': int(len(work.columns))},
    'kpis': kpis,
    'distribuciones': {
        'tipo': top_counts('tipo'),
        'provincia': top_counts('nombre_provincia'),
        'canton': top_counts('nombre_canton'),
        'distrito': top_counts('nombre_distrito'),
        'infraccion': top_counts('presunta_infraccion'),
        'arma': top_counts('arma'),
    },
    'mapaCalor': heat.head(350).rename(columns={'lat_grid':'lat','lng_grid':'lng'}).to_dict(orient='records')
}

Path('analisis_detenidos_resumen_notebook.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary['kpis']